# NB09 - Consolidate + pair-level split

**Run once, after all six generators finish.** Reconciles the true counts from the shards, removes any accidental duplicates, and builds a deterministic **pair-level** train/val/test split (70/15/15) where every real image and its six AI partners stay in the same split (so a model can never see a scene in training and again in test).

Outputs written to the repo: `manifest.parquet` (one row per image, with its split) and `splits.parquet` (source_real_id -> split). The image shards in `real/` and `data/*/` are left as-is; the manifest is the index over them.

GPU not needed. Internet ON.

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
# diffusers stack + hub/io. sentencepiece+protobuf are needed by the T5 text encoders (Flux/PixArt).
pip("diffusers>=0.30", "transformers>=4.40", "accelerate", "safetensors",
    "sentencepiece", "protobuf", "huggingface_hub>=0.23", "pyarrow", "pillow")
print("deps installed")

In [ ]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN","HUGGINGFACE_TOKEN","HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
print("loaded. generators:", list(cfg["generators"]))

In [ ]:
# 1. gather metadata (no image bytes) from every shard
import hashlib
from collections import Counter, defaultdict

prefixes = ["real/"] + [f"data/{m}/" for m in cfg["generators"]]
rows = []          # (image_id, source_real_id, generator, label, sha256)
for p in prefixes:
    shards = C.list_shards(REPO_ID, p, TOKEN)
    cnt = 0
    for f in shards:
        loc = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t = C.pq.read_table(loc, columns=["image_id","source_real_id","generator","label","sha256"])
        rows += list(zip(t.column("image_id").to_pylist(), t.column("source_real_id").to_pylist(),
                         t.column("generator").to_pylist(), t.column("label").to_pylist(),
                         t.column("sha256").to_pylist()))
        cnt += t.num_rows
    print(f"{p:16} {cnt} rows in {len(shards)} shards")
print("total rows gathered:", len(rows))

In [ ]:
# 2. dedup by image_id (idempotent seeding means a double-run makes identical rows);
#    flag identical bytes shared across different ids (a real==AI collision would be a leak)
seen_id, uniq = set(), []
dup_ids = 0
for r in rows:
    if r[0] in seen_id:
        dup_ids += 1; continue
    seen_id.add(r[0]); uniq.append(r)

sha_to_ids = defaultdict(set)
for iid, sid, gen, lab, sha in uniq:
    sha_to_ids[sha].add(iid)
cross = {s: ids for s, ids in sha_to_ids.items() if len(ids) > 1}
print(f"unique images: {len(uniq)} | duplicate image_id rows dropped: {dup_ids}")
print(f"identical-byte groups (review if many): {len(cross)}")
for s, ids in list(cross.items())[:3]:
    print("   shared bytes:", list(ids)[:4])

In [ ]:
# 3. deterministic pair-level split on source_real_id (whole pair group share one split)
sp = cfg["split"]; seed = sp.get("seed", 1234)
def assign_split(src_id):
    h = int(hashlib.sha256(f"{seed}:{src_id}".encode()).hexdigest()[:8], 16)
    r = (h % 100000) / 100000.0
    return "train" if r < sp["train"] else ("val" if r < sp["train"] + sp["val"] else "test")

split_of = {}
for _, sid, _, _, _ in uniq:
    if sid not in split_of:
        split_of[sid] = assign_split(sid)

# manifest rows with split; splits table keyed by source_real_id
man = [dict(image_id=iid, source_real_id=sid, generator=gen, label=int(lab),
            sha256=sha, split=split_of[sid]) for (iid, sid, gen, lab, sha) in uniq]

from collections import Counter
print("split sizes (images):", Counter(m["split"] for m in man))
print("pairs (distinct source_real_id):", len(split_of),
      "| per split:", Counter(split_of.values()))
print("per generator:", Counter(m["generator"] for m in man))

In [ ]:
# 4. write manifest.parquet + splits.parquet to the repo
import pyarrow as pa, tempfile, os
man_schema = pa.schema([("image_id", pa.string()), ("source_real_id", pa.string()),
                        ("generator", pa.string()), ("label", pa.int8()),
                        ("sha256", pa.string()), ("split", pa.string())])
man_tbl = pa.Table.from_pylist(man, schema=man_schema)
split_tbl = pa.Table.from_pylist(
    [{"source_real_id": k, "split": v} for k, v in split_of.items()],
    schema=pa.schema([("source_real_id", pa.string()), ("split", pa.string())]))

api = C.HfApi(token=TOKEN)
for tbl, name in [(man_tbl, "manifest.parquet"), (split_tbl, "splits.parquet")]:
    tmp = os.path.join(tempfile.gettempdir(), name)
    C.pq.write_table(tbl, tmp, compression="zstd")
    api.upload_file(path_or_fileobj=tmp, path_in_repo=name, repo_id=REPO_ID,
                    repo_type="dataset", commit_message=f"write {name}")
    print("uploaded", name, tbl.num_rows, "rows")
print("\nNB09 complete. Next: NB10 (validation + leak audit).")